# PSO-ARM — Penentuan Threshold Optimal (Support & Confidence)

Notebook ini menjalankan **Particle Swarm Optimization (PSO)** untuk menentukan nilai
**minimum support** dan **minimum confidence** yang optimal bagi Association Rule Mining
(FP-Growth), mengikuti pendekatan Das et al. (2023).

Arsitektur dua tahap:
1. **Tahap ini (PSO-ARM):** mencari threshold optimal secara global pada data proyek IT & R&D.
2. **Tahap lanjut (FP-Growth berpartisi):** threshold hasil PSO diterapkan pada mining
   per kombinasi domain × level risiko (notebook terpisah).

> Catatan reproducibility: seluruh proses memakai `RANDOM_SEED` sehingga hasilnya konsisten
> setiap kali dijalankan pada data yang sama.

## 1. Import Library & Konfigurasi

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# =========================== KONFIGURASI ===========================
# Path dataset mentah (arahkan ke file yang SAMA dengan notebook FP-Growth utama).
# Boleh .csv atau .xlsx; kode akan memfilter proyek IT & R&D secara otomatis.
DATASET_PATH = "project_risk_raw_dataset(asli).xlsx"

# Reproducibility
RANDOM_SEED = 42

# Batas panjang itemset saat evaluasi fitness.
#   None  -> tanpa batas (SETIA dengan eksperimen awal; menghasilkan ~6,1% / 50%)
#   3     -> dibatasi 3 (lebih cepat, namun optimum bergeser ke support lebih tinggi)
MAX_LEN = None

# Imputasi KNN pada tahap PSO.
#   False -> mengikuti eksperimen awal (threshold operasional 6,1% / 50%)
#   True  -> menyamakan dengan pipeline utama; CATATAN: dapat menggeser threshold optimal
APPLY_IMPUTATION = False
# ===================================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Library siap. SEED =", RANDOM_SEED, "| MAX_LEN =", MAX_LEN, "| APPLY_IMPUTATION =", APPLY_IMPUTATION)

ModuleNotFoundError: No module named 'mlxtend'

## 2. Pra-pemrosesan Data

Tahapan: **filter IT & R&D → hapus `Project_ID` → (opsional) imputasi KNN →
diskritisasi Equal-Frequency Binning (3 bin) → One-Hot Encoding**.

Deteksi tipe kolom memakai `is_numeric_dtype` agar kompatibel lintas versi pandas.

In [ ]:
print("Memuat & memfilter dataset (IT & R&D)...")

def load_dataset(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)

def non_numeric_cols(frame):
    return [c for c in frame.columns if not pd.api.types.is_numeric_dtype(frame[c])]

df_raw = load_dataset(DATASET_PATH)
df = df_raw[df_raw["Project_Type"].isin(["IT", "R&D"])].copy()
print(f"   -> Jumlah record IT & R&D : {df.shape[0]} baris")

# Hapus kolom identifier
if "Project_ID" in df.columns:
    df = df.drop(columns=["Project_ID"])

# Normalisasi artefak 'Excel-time' pada kolom teks, lalu paksa numerik bila memungkinkan
def fix_excel_time_to_float(val):
    if isinstance(val, str) and ":" in val:
        try:
            parts = val.split(":")
            return float(f"{parts[0]}.{parts[1]}")
        except Exception:
            return val
    return val

for col in non_numeric_cols(df):
    df[col] = df[col].apply(fix_excel_time_to_float)
    try:
        df[col] = pd.to_numeric(df[col])   # hanya berhasil jika seluruh nilai numerik
    except (ValueError, TypeError):
        pass

# --- (Opsional) KNN Imputation (k=5) ---
if APPLY_IMPUTATION:
    total_before = int(df.isnull().sum().sum())
    print(f"   -> Sel kosong sebelum imputasi : {total_before}")
    work = df.copy()
    encoders = {}
    for col in non_numeric_cols(work):
        le = LabelEncoder()
        mask = work[col].notnull()
        le.fit(work.loc[mask, col].astype(str))
        codes = pd.Series(np.nan, index=work.index, dtype="float64")
        codes.loc[mask] = le.transform(work.loc[mask, col].astype(str))
        work[col] = codes                     # ganti seluruh kolom -> numerik (NaN dipertahankan)
        encoders[col] = le
    work = work.apply(pd.to_numeric, errors="coerce")
    imputed = pd.DataFrame(
        KNNImputer(n_neighbors=5).fit_transform(work),
        columns=work.columns, index=work.index,
    )
    for col, le in encoders.items():
        vals = imputed[col].round().astype(int).clip(0, len(le.classes_) - 1)
        imputed[col] = le.inverse_transform(vals)
    df = imputed
    print(f"   -> Sel kosong sesudah imputasi : {int(df.isnull().sum().sum())}")
else:
    print("   -> Imputasi dilewati (mengikuti eksperimen awal).")

# --- Diskritisasi Equal-Frequency Binning (3 bin) pada kolom numerik ---
for col in [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]:
    try:
        labels = [f"{col}_Low", f"{col}_Medium", f"{col}_High"]
        df[col] = pd.qcut(df[col], q=3, labels=labels, duplicates="drop")
    except ValueError:
        df[col] = df[col].astype(str)   # jika nilai terlalu seragam

# --- One-Hot Encoding (boolean; disyaratkan mlxtend) ---
df_encoded = pd.get_dummies(df, dtype=bool)
print(f"   -> Dimensi akhir setelah encoding : {df_encoded.shape[1]} kolom")
print("Pra-pemrosesan selesai.")

## 3. Fungsi Fitness (Kung's Fitness Function)

Setiap partikel merepresentasikan kandidat pasangan *(min_support, min_confidence)*.
Kualitasnya diukur dengan **Kung's fitness** (Das et al., 2023, pers. 9):

$$Fitness(R) = confidence(R)\times \log_{10}\big(support(R)\times length(R) + 1\big)$$

Skor partikel = **rata-rata** fitness seluruh aturan yang **berujung pada `Risk_Level`**
(consequent mengandung level risiko). Nilai dikembalikan 0 bila tidak ada aturan yang lolos.

In [ ]:
def _association_rules_compat(itemsets, min_confidence):
    """Pemanggilan association_rules yang kompatibel lintas versi mlxtend."""
    try:
        return association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    except TypeError:
        # mlxtend >= 0.23 menambahkan argumen num_itemsets
        return association_rules(itemsets, num_itemsets=len(df_encoded),
                                 metric="confidence", min_threshold=min_confidence)

def calculate_kungs_fitness(min_support, min_confidence):
    try:
        frequent_itemsets = fpgrowth(
            df_encoded, min_support=min_support, use_colnames=True, max_len=MAX_LEN
        )
        if frequent_itemsets.empty:
            return 0.0

        rules = _association_rules_compat(frequent_itemsets, min_confidence)
        if rules.empty:
            return 0.0

        # Hanya aturan yang consequent-nya mengandung Risk_Level (target)
        risk_rules = rules[rules["consequents"].apply(
            lambda x: any("Risk_Level" in str(item) for item in x)
        )]
        if risk_rules.empty:
            return 0.0

        length = risk_rules["antecedents"].apply(len) + risk_rules["consequents"].apply(len)
        fitness = risk_rules["confidence"] * np.log10(risk_rules["support"] * length + 1)
        return float(fitness.mean())
    except Exception:
        return 0.0

# Cache: PSO sering mengevaluasi posisi yang mirip; hasil di-cache agar tidak dihitung ulang.
_fitness_cache = {}
def fitness_cached(min_support, min_confidence):
    key = (round(float(min_support), 4), round(float(min_confidence), 4))
    if key not in _fitness_cache:
        _fitness_cache[key] = calculate_kungs_fitness(*key)
    return _fitness_cache[key]

print("Fungsi fitness siap.")

## 4. Particle Swarm Optimization

PSO standar (metode *inertia weight*), dilengkapi **early-stopping**: pencarian berhenti bila
`Gbest` tidak membaik selama `PATIENCE` iterasi berturut-turut. Riwayat tiap iterasi disimpan
untuk plot konvergensi.

In [ ]:
# --- Parameter PSO ---
NUM_PARTICLES  = 10
NUM_ITERATIONS = 15
W  = 0.5      # inertia weight
C1 = 1.5      # cognitive constant (tarikan ke Pbest)
C2 = 1.5      # social constant   (tarikan ke Gbest)

# Ruang pencarian [min, max]
BOUNDS = np.array([[0.05, 0.40],    # min_support
                   [0.50, 0.95]])   # min_confidence

PATIENCE = 5          # iterasi tanpa perbaikan sebelum berhenti
TOL = 1e-6

rng = np.random.default_rng(RANDOM_SEED)
positions  = rng.uniform(BOUNDS[:, 0], BOUNDS[:, 1], (NUM_PARTICLES, 2))
velocities = np.zeros((NUM_PARTICLES, 2))

pbest_positions = positions.copy()
pbest_fitness   = np.zeros(NUM_PARTICLES)
gbest_position  = np.zeros(2)
gbest_fitness   = 0.0

history = []          # (iterasi, gbest_fitness, gbest_support, gbest_confidence)
stall, last_gbest = 0, -1.0

print("Memulai pencarian threshold optimal (PSO-ARM)...\n")
for it in range(NUM_ITERATIONS):
    # Evaluasi & update Pbest/Gbest
    for j in range(NUM_PARTICLES):
        fit = fitness_cached(positions[j, 0], positions[j, 1])
        if fit > pbest_fitness[j]:
            pbest_fitness[j] = fit
            pbest_positions[j] = positions[j]
        if fit > gbest_fitness:
            gbest_fitness = fit
            gbest_position = positions[j].copy()

    # Update kecepatan & posisi
    for j in range(NUM_PARTICLES):
        r1, r2 = rng.random(2)
        velocities[j] = (W * velocities[j]
                         + C1 * r1 * (pbest_positions[j] - positions[j])
                         + C2 * r2 * (gbest_position - positions[j]))
        positions[j] = np.clip(positions[j] + velocities[j], BOUNDS[:, 0], BOUNDS[:, 1])

    history.append((it + 1, gbest_fitness, gbest_position[0], gbest_position[1]))
    print(f"Iterasi {it+1:>2}/{NUM_ITERATIONS} | Gbest Fitness: {gbest_fitness:.4f} "
          f"| Support: {gbest_position[0]:.4f} | Confidence: {gbest_position[1]:.4f}")

    # Early stopping
    stall = stall + 1 if abs(gbest_fitness - last_gbest) < TOL else 0
    last_gbest = gbest_fitness
    if stall >= PATIENCE:
        print(f"\n[KONVERGEN] Gbest stabil {PATIENCE} iterasi berturut-turut. Berhenti di iterasi {it+1}.")
        break

print("\nPencarian selesai.")

## 5. Hasil & Visualisasi Konvergensi

In [ ]:
opt_support, opt_confidence = gbest_position
print("=" * 56)
print("HASIL PSO-ARM — Threshold Optimal Terpilih")
print("=" * 56)
print(f"  Min Support    : {opt_support:.4f}   (~{opt_support*100:.2f}%)")
print(f"  Min Confidence : {opt_confidence:.4f}   (~{opt_confidence*100:.2f}%)")
print(f"  Gbest Fitness  : {gbest_fitness:.4f}")
print("=" * 56)

hist = np.array(history, dtype=float)
iters = hist[:, 0]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(iters, hist[:, 1], marker="o", color="#1f77b4", label="Gbest Fitness")
ax1.set_xlabel("Iterasi")
ax1.set_ylabel("Gbest Fitness", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.set_title("Konvergensi PSO-ARM: Fitness & Threshold Optimal")

ax2 = ax1.twinx()
ax2.plot(iters, hist[:, 2], marker="s", linestyle="--", color="#d62728", label="Support")
ax2.plot(iters, hist[:, 3], marker="^", linestyle=":",  color="#2ca02c", label="Confidence")
ax2.set_ylabel("Nilai Threshold")

l1, lab1 = ax1.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lab1 + lab2, loc="center right")

fig.tight_layout()
plt.savefig("konvergensi_pso_arm.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot konvergensi disimpan: 'konvergensi_pso_arm.png'")

## 6. (Opsional) Basis Aturan pada Threshold Optimal — *sanity check*

Verifikasi cepat: menampilkan beberapa aturan `{kondisi} → {Risk_Level}` terkuat pada
threshold hasil PSO. **Catatan:** ini mining global; mining per-partisi (8 kombinasi
domain × level) dilakukan pada notebook FP-Growth utama.

In [ ]:
freq = fpgrowth(df_encoded, min_support=opt_support, use_colnames=True, max_len=MAX_LEN)
rules = _association_rules_compat(freq, opt_confidence)
risk_rules = rules[rules["consequents"].apply(
    lambda x: any("Risk_Level" in str(item) for item in x)
)].sort_values(by=["lift", "confidence"], ascending=[False, False])

print(f"Jumlah aturan ber-consequent Risk_Level pada threshold optimal: {len(risk_rules)}\n")
print("5 aturan teratas (berdasarkan lift):")
for _, row in risk_rules.head(5).iterrows():
    ante = ", ".join(list(row["antecedents"]))
    cons = ", ".join(list(row["consequents"]))
    print(f"\nIF   [{ante}]")
    print(f"THEN [{cons}]")
    print(f"     support={row['support']:.4f} | confidence={row['confidence']:.4f} | lift={row['lift']:.4f}")